# 12. Router Knowledge Base — 질문을 알맞은 지식원으로 보내기

## 학습 목표

- router와 handoff/subagent의 차이를 구분합니다.
- 여러 지식원을 source별 retriever처럼 다루는 구조를 만듭니다.
- 라우팅 결과를 간단한 테스트 케이스로 검증합니다.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

In [ ]:
# LangSmith / Langfuse 설정 — 키가 없으면 비활성 상태로 둡니다.
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGSMITH_PROJECT", "agent-notebooks")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

## 12.1 지식원 정의

지식원이 많아질수록 하나의 retriever보다 먼저 source를 고르는 router가 유용합니다.

In [ ]:
knowledge_sources = {
    "billing": ["환불은 결제일로부터 7일 이내 가능합니다."],
    "technical": ["오류 신고에는 로그와 재현 단계가 필요합니다."],
    "product": ["Deep Agents는 planning, files, subagents를 포함합니다."],
}

list(knowledge_sources)

## 12.2 deterministic router

실제 서비스에서는 structured output router를 쓰되, 먼저 규칙 기반 router로 테스트 모양을 고정합니다.

In [ ]:
def route_source(question: str) -> str:
    q = question.lower()
    if "환불" in q or "refund" in q:
        return "billing"
    if "오류" in q or "error" in q:
        return "technical"
    return "product"

route_source("Deep Agents 기능은?")

## 12.3 source-local search

라우터가 고른 source 안에서만 검색하면 context가 작아지고 근거 관리가 쉬워집니다.

In [ ]:
def retrieve(question: str) -> dict:
    source = route_source(question)
    docs = knowledge_sources[source]
    return {"source": source, "documents": docs}

retrieve("환불 조건 알려줘")

## 12.4 라우터 평가

라우터는 답변 품질과 별도로 source 선택 정확도를 평가합니다.

In [ ]:
cases = [
    ("환불 가능한가요?", "billing"),
    ("앱 오류가 납니다", "technical"),
    ("Deep Agents가 뭐예요?", "product"),
]

for question, expected in cases:
    actual = route_source(question)
    print(question, actual, actual == expected)

---

## 정리

| 항목 | 내용 |
|---|---|
| **다룬 기술** | router, source-local retrieval, routing eval |
| **핵심 개념** | Router는 답변을 생성하기 전에 “어디에서 찾을지”를 결정하는 계층입니다. |
| **다음 단계** | `13_skills_sql_assistant.ipynb` 후보 |

**참고 문서:**
- `docs/langchain/multi-agent/router.md`
- `docs/langchain/multi-agent/router-knowledge-base.md`
- `docs/langchain/structured-output.md`